アヤメの花に関するデータセットを使って，授業で作ったTwoLayerNetを学習し，アヤメの種類を言い当てられるようにしてください．

授業のときはMNISTのデータでしたので学習に時間がかかりましたが，アヤメの花のデータであれば4次元のデータなので学習の計算時間はそれほどかからないと思います．

In [1]:
# アヤメのデータセットをインターネットからロードする
from sklearn import datasets

iris = datasets.load_iris()
x = iris.data
y = iris.target

In [2]:
# xはアヤメのがくの長さ、幅、花弁の長さ、幅を持ちます
x[:3]

array([[5.1, 3.5, 1.4, 0.2],
       [4.9, 3. , 1.4, 0.2],
       [4.7, 3.2, 1.3, 0.2]])

In [3]:
# サイズは150です (150のアヤメのデータを持ちます)
len(x)

150

In [4]:
# アヤメの種類は3種類あり，種類ごとに，0: Setosa, 1: Versicolor, 2: Virginicaというように3つの数字のラベルが付いています
y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [5]:
# アヤメのがくの長さ、幅、花弁の長さ、幅を標準化 (平均値0、標準偏差1) します．
# ここで正規化(0～1など)していただいても良いです．
from sklearn import preprocessing

x = preprocessing.scale(x)

In [6]:
# ラベルは0～2の値になっているのでone-hot表現に変換します
from keras.utils import to_categorical

y = to_categorical(y)

In [7]:
# 訓練データとテストデータに分割します．
from sklearn.model_selection import train_test_split

x_train, x_test, t_train, t_test = train_test_split(x, y, random_state=0)


In [21]:
def numerical_gradient(f, X):
    if X.ndim == 1:
        return _numerical_gradient_no_batch(f, X)
    else:
        grad = np.zeros_like(X)

        for idx, x in enumerate(X):
            grad[idx] = _numerical_gradient_no_batch(f, x)

        return grad

def softmax(a):
    c = np.max(a)
    exp_a = np.exp(a - c) # オーバーフロー対策
    sum_exp_a = np.sum(exp_a)
    y = exp_a / sum_exp_a

    return y
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def _numerical_gradient_no_batch(f, x):
    h = 1e-4 # 0.0001
    grad = np.zeros_like(x) # xと同じ形状の配列を生成

    for idx in range(x.size):
        tmp_val = x[idx] # ターゲットとする変数を退避しておく

        # f(x+h)の計算
        x[idx] = tmp_val + h
        fxh1 = f(x)

        # f(x-h)の計算
        x[idx] = tmp_val - h
        fxh2 = f(x)

        grad[idx] = (fxh1 - fxh2) / (2*h)

        x[idx] = tmp_val # 退避しておいた値でもとの値に戻す

    return grad

def cross_entropy_error(y, t):
    if y.ndim == 1:
        t = t.reshape(1, t.size)
        y = y.reshape(1, y.size)

    batch_size = y.shape[0]
    return -np.sum(t * np.log(y + 1e-7)) / batch_size

In [22]:
class TwoLayerNet:

    def __init__(self, input_size, hidden_size, output_size, weight_init_std=0.01):
        # 重みの初期化
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size)
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = weight_init_std * np.random.randn(hidden_size, output_size)
        self.params['b2'] = np.zeros(output_size)

    def predict(self, x):
        W1, W2 = self.params['W1'], self.params['W2']
        b1, b2 = self.params['b1'], self.params['b2']

        a1 = np.dot(x, W1) + b1
        z1 = sigmoid(a1)
        a2 = np.dot(z1, W2) + b2
        y = softmax(a2)

        return y

    # x:入力データ, t:教師データ
    def loss(self, x, t):
        y = self.predict(x)

        return cross_entropy_error(y, t)

    def accuracy(self, x, t):
        y = self.predict(x)
        y = np.argmax(y, axis=1)
        t = np.argmax(t, axis=1)

        accuracy = np.sum(y == t) / float(x.shape[0])
        return accuracy

    # x:入力データ, t:教師データ
    def numerical_gradient(self, x, t):
        loss_W = lambda W: self.loss(x, t)

        grads = {}
        grads['W1'] = numerical_gradient(loss_W, self.params['W1'])
        grads['b1'] = numerical_gradient(loss_W, self.params['b1'])
        grads['W2'] = numerical_gradient(loss_W, self.params['W2'])
        grads['b2'] = numerical_gradient(loss_W, self.params['b2'])

        return grads

In [32]:
import numpy as np
# t_train = np.eye(10)[t_train]   # 単位行列（対角成分が1の行列）を作成し，インデックスを指定
# t_test = np.eye(10)[t_test]   # 単位行列（対角成分が1の行列）を作成し，インデックスを指定

network = TwoLayerNet(input_size=4, hidden_size=50, output_size=3)

# iters_num = 10000  # 繰り返しの回数を適宜設定する
# batch_size = 100

iters_num = 1000
batch_size = 50

train_size = x_train.shape[0]
learning_rate = 0.08

train_loss_list = []
iter_per_epoch = max(train_size // batch_size, 1)

for i in range(iters_num):
    batch_mask = np.random.choice(train_size, batch_size)
    x_batch = x_train[batch_mask]
    t_batch = t_train[batch_mask]

    # 勾配の計算
    grad = network.numerical_gradient(x_batch, t_batch)

    # パラメータの更新
    for key in ('W1', 'b1', 'W2', 'b2'):
        network.params[key] -= learning_rate * grad[key]

    loss = network.loss(x_batch, t_batch)
    train_loss_list.append(loss)
    # print(loss)

    if i % iter_per_epoch == 0:
        train_acc = network.accuracy(x_train, t_train)
        test_acc = network.accuracy(x_test, t_test)
        print("train acc, test acc | " + str(train_acc) + ", " + str(test_acc))

train acc, test acc | 0.36607142857142855, 0.23684210526315788
train acc, test acc | 0.36607142857142855, 0.23684210526315788
train acc, test acc | 0.36607142857142855, 0.23684210526315788
train acc, test acc | 0.36607142857142855, 0.23684210526315788
train acc, test acc | 0.6964285714285714, 0.5789473684210527
train acc, test acc | 0.36607142857142855, 0.23684210526315788
train acc, test acc | 0.36607142857142855, 0.23684210526315788
train acc, test acc | 0.36607142857142855, 0.23684210526315788
train acc, test acc | 0.36607142857142855, 0.23684210526315788
train acc, test acc | 0.33035714285714285, 0.34210526315789475
train acc, test acc | 0.36607142857142855, 0.23684210526315788
train acc, test acc | 0.36607142857142855, 0.23684210526315788
train acc, test acc | 0.36607142857142855, 0.23684210526315788
train acc, test acc | 0.36607142857142855, 0.23684210526315788
train acc, test acc | 0.33035714285714285, 0.34210526315789475
train acc, test acc | 0.33035714285714285, 0.342105263157

In [33]:
network.accuracy(x_test, t_test)

np.float64(0.9736842105263158)

以上で訓練データとテストデータを得られますので，授業で作ったニューラルネットワークのプログラムを使ってニューラルネットワークの学習を行い，その結果の精度を算出してください．精度の計算は
```
network.accuracy(x_test, t_test)
```
でできます．0.9以上になるように```lerning_rate```や```batch_size```を調節してみてください．繰り返し回数```iters_num```も適当に設定して構いません．

